# Interrupted Time Series (ITS) / Segmented Regression

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Setup: shared imports and data generators

This cell reproduces the book's `_common.py` so the notebook runs standalone.

In [ ]:
"""Shared setup for all chapters: imports, plotting theme, and data generators."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def make_panel(
    n_stores=120,
    n_markets=12,
    n_weeks=104,                # 2 years of weekly data
    treat_markets=(0, 1, 2, 3), # which markets get the program
    treat_week=60,              # program launch week
    true_effect=8.0,            # true lift in weekly revenue ($k)
    effect_ramp=6,              # weeks for the effect to reach full size
    seed=42,
):
    """NorthShore Retail store-week panel with known treatment effect."""
    rng = np.random.default_rng(seed)
    stores = np.arange(n_stores)
    markets = stores % n_markets
    store_fe = rng.normal(100, 15, n_stores)
    market_fe = rng.normal(0, 8, n_markets)
    store_trend = rng.normal(0.05, 0.03, n_stores)

    rows = []
    market_shock = rng.normal(0, 3, (n_markets, n_weeks))
    for w in range(n_weeks):
        season = 6 * np.sin(2 * np.pi * w / 52)
        for s in stores:
            m = markets[s]
            treated_unit = m in treat_markets
            post = w >= treat_week
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp
            else:
                tau = 0.0
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    return pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])


def make_customers(n=8000, true_att=12.0, seed=42):
    """Loyalty-program adoption data with selection on observables."""
    rng = np.random.default_rng(seed)
    tenure = rng.gamma(3, 1.2, n)
    spend0 = rng.lognormal(4.0, 0.5, n)
    engage = rng.beta(2, 5, n) * 10
    urban = rng.binomial(1, 0.55, n)
    logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
    p = 1 / (1 + np.exp(-logit))
    D = rng.binomial(1, p)
    y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
         + true_att * D + rng.normal(0, 12, n))
    return pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                             urban=urban, D=D, y=y))


panel = make_panel()
cust = make_customers()
X_cols = ["tenure", "spend0", "engage", "urban"]
TRUE_ATT = 12.0


**Business setting:** a company-wide intervention with **no control group at all** — e.g., a sitewide checkout redesign, a national price increase, a new returns policy. All you have is the outcome series before and after.

## Logic and model

ITS asks whether the series exhibits a **level change** and/or **slope change** at the intervention point, relative to the projection of the pre-period model:

$$
Y_t = \beta_0 + \beta_1 t + \beta_2 \, \text{post}_t + \beta_3 \,(t - t^*)\,\text{post}_t + \text{seasonality} + \varepsilon_t
$$

- $\beta_2$: immediate level shift at the intervention.
- $\beta_3$: change in trend after the intervention.

**Identifying assumption:** absent the intervention, the pre-period trend and seasonal structure would have continued. Anything *else* that changed at the same time (a competitor exit, a holiday, a pandemic) is absorbed into the "effect."

## Simulated example: national price increase

In [ ]:
n = 156   # 3 years of weeks
t = np.arange(n)
t_star = 110
season = 5 * np.sin(2 * np.pi * t / 52)
y0 = 200 + 0.30 * t + season + rng.normal(0, 3, n)   # counterfactual
level_drop, slope_change = -9.0, -0.10               # price hike hurts volume
y = y0 + (t >= t_star) * (level_drop + slope_change * (t - t_star))
its = pd.DataFrame({"t": t, "y": y, "post": (t >= t_star).astype(int)})
its["t_post"] = np.maximum(0, its.t - t_star)
its["sin52"] = np.sin(2 * np.pi * its.t / 52)
its["cos52"] = np.cos(2 * np.pi * its.t / 52)

## Estimation with autocorrelation-robust errors

Time-series residuals are autocorrelated; OLS SEs are too small. Use Newey–West (HAC) errors, or model the errors explicitly (GLS/ARIMA).

In [ ]:
m_its = smf.ols("y ~ t + post + t_post + sin52 + cos52", data=its).fit(
    cov_type="HAC", cov_kwds={"maxlags": 8})
print(m_its.summary().tables[1])

Check residual autocorrelation and consider an ARIMA error structure if HAC seems insufficient:

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(m_its.resid, lags=30); plt.show()
print(acorr_ljungbox(m_its.resid, lags=[8, 16], return_df=True))

## Visualizing the counterfactual

In [ ]:
cf = m_its.params["Intercept"] + m_its.params["t"] * its.t \
     + m_its.params["sin52"] * its.sin52 + m_its.params["cos52"] * its.cos52
plt.figure(figsize=(10, 4))
plt.plot(its.t, its.y, label="Observed")
plt.plot(its.t[its.post == 1], cf[its.post == 1], "--",
         label="Projected counterfactual")
plt.axvline(t_star, color="red", ls=":")
plt.legend(); plt.tight_layout(); plt.show()

## Strengthening an ITS

A bare ITS is among the weakest designs here. Strengthen it by:

- **Adding a control series** (a comparable product line, region, or competitor metric not exposed to the intervention) → a *controlled ITS*, which is essentially DiD with rich time structure.
- **Placebo dates:** re-estimate at fake intervention dates in the pre-period; significant "effects" indicate model misspecification.
- **Long pre-period** covering at least 2 full seasonal cycles, so seasonality is estimated, not confused with the effect.
- **Checking for co-interventions:** maintain an intervention calendar; the most common ITS failure in industry is a forgotten concurrent change (a promo, a stockout, a tracking change).